# Derivative Pricing with Black-Scholes inputs and historical volatility

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from DerivModel import train_model, FeedForwardNetwork, black_scholes
from DerivMetrics import evaluate_metrics, compute_metrics 
from DerivPlots import scatter_plot, plot_errors
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange

## 20-day Historical Vol

### Load and Prepare the dataframe

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
#Separate into X and y
X = dfopt[['type', 'strike', '20D_HV_x', 'maturity', 'rate', 'underlying']]
y = dfopt['mid']

In [ ]:
X['type'] = X['type'].replace({'call': 1, 'put': 0})

In [ ]:
# Split data into train and a temporary set (tmp) first
X_train, us_test, y_train, y_test = train_test_split(X, y, test_size=0.2) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(us_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

### Build and Train model

In [ ]:
epochs = 50
no_layers = 5
no_neurons = 4096
lr = 0.001

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
plot_file = 'bs_hist20.png'
plot_errors(train_errors, test_errors, plot_file)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

In [ ]:
metric_file = 'bsmetrics_hist20.json'
with open(metric_file, 'w') as json_file:
    json.dump(metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist20.png'
scatter_plot(all_results, all_targets, scatter_filename)

In [ ]:
bs_lst = list()
for _, row in us_test.iterrows():
    bsres = black_scholes(row['type'], row['strike'], row['20D_HV_x'],
                          row['maturity'], row['underlying'], row['rate'])
    bs_lst.append(bsres.value())

In [ ]:
bs_metrics = compute_metrics(torch.tensor(bs_lst), torch.tensor(y_test.values))

In [ ]:
bs_metrics

In [ ]:
metric_file = 'bsmetrics_hist20_BS.json'
with open(metric_file, 'w') as json_file:
    json.dump(bs_metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist20_BS.png'
scatter_plot(torch.tensor(bs_lst), torch.tensor(y_test.values), scatter_filename)

## 40-day Historical Vol

In [ ]:
#Separate into X and y
X = dfopt[['type', 'strike', '40D_HV_x', 'maturity', 'rate', 'underlying']]
y = dfopt['mid']

X['type'] = X['type'].replace({'call': 1, 'put': 0})

# Split data into train and a temporary set (tmp) first
X_train, us_test, y_train, y_test = train_test_split(X, y, test_size=0.2) 

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(us_test)

torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

### Build and Train model

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
plot_file = 'bs_hist40.png'
plot_errors(train_errors, test_errors, plot_file)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

In [ ]:
metric_file = 'bsmetrics_hist40.json'
with open(metric_file, 'w') as json_file:
    json.dump(metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist40.png'
scatter_plot(all_results, all_targets, scatter_filename)

In [ ]:
bs_lst = list()
for _, row in us_test.iterrows():
    bsres = black_scholes(row['type'], row['strike'], row['40D_HV_x'],
                          row['maturity'], row['underlying'], row['rate'])
    bs_lst.append(bsres.value())

bs_metrics = compute_metrics(torch.tensor(bs_lst), torch.tensor(y_test.values))

In [ ]:
bs_metrics

In [ ]:
metric_file = 'bsmetrics_hist40_BS.json'
with open(metric_file, 'w') as json_file:
    json.dump(bs_metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist40_BS.png'
scatter_plot(torch.tensor(bs_lst), torch.tensor(y_test.values), scatter_filename)

# 60-day Historical Vol

In [ ]:
#Separate into X and y
dfopt = dfopt.dropna()
X = dfopt[['type', 'strike', '60D_HV_x', 'maturity', 'rate', 'underlying']]
y = dfopt['mid']

X['type'] = X['type'].replace({'call': 1, 'put': 0})

# Split data into train and a temporary set (tmp) first
X_train, us_test, y_train, y_test = train_test_split(X, y, test_size=0.2) 

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(us_test)

torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

## Build and Train model

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
plot_file = 'bs_hist60.png'
plot_errors(train_errors, test_errors, plot_file)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

In [ ]:
metric_file = 'bsmetrics_hist60.json'
with open(metric_file, 'w') as json_file:
    json.dump(metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist60.png'
scatter_plot(all_results, all_targets, scatter_filename)

In [ ]:
bs_lst = list()
for _, row in us_test.iterrows():
    bsres = black_scholes(row['type'], row['strike'], row['60D_HV'],
                          row['maturity'], row['underlying'], row['rate'])
    bs_lst.append(bsres.value())

bs_metrics = compute_metrics(torch.tensor(bs_lst), torch.tensor(y_test.values))

In [ ]:
bs_metrics

In [ ]:
metric_file = 'bsmetrics_hist60_BS.json'
with open(metric_file, 'w') as json_file:
    json.dump(bs_metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist60_BS.png'
scatter_plot(torch.tensor(bs_lst), torch.tensor(y_test.values), scatter_filename)